In [1]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta


In [2]:

fake = Faker()
np.random.seed(42)
random.seed(42)

# ----------------------------
# CONFIGURATION
# ----------------------------
NUM_STORES = 5
NUM_PRODUCTS = 200
NUM_CUSTOMERS = 3000
NUM_EMPLOYEES = 80
START_DATE = datetime(2024, 1, 1)
END_DATE = datetime(2024, 12, 31)

# ----------------------------
# 1. STORE DATA
# ----------------------------
stores = []
for i in range(1, NUM_STORES + 1):
    stores.append({
        "store_id": f"S{i}",
        "store_size_sqft": np.random.randint(2000, 8000),
        "rent_cost": np.random.randint(50000, 150000),
        "fixed_operating_cost": np.random.randint(100000, 300000)
    })

df_stores = pd.DataFrame(stores)

# ----------------------------
# 2. PRODUCT MASTER DATA
# ----------------------------
categories = ["Grocery", "Electronics", "Clothing", "Home", "Beauty"]

products = []
for i in range(1, NUM_PRODUCTS + 1):
    cost = round(np.random.uniform(5, 200), 2)
    margin = np.random.uniform(0.1, 0.4)
    price = round(cost * (1 + margin), 2)

    products.append({
        "sku": f"P{i}",
        "category": random.choice(categories),
        "cost_price": cost,
        "selling_price": price,
        "supplier_lead_time_days": np.random.randint(3, 20)
    })

df_products = pd.DataFrame(products)


In [4]:
import pandas as pd
import numpy as np
from datetime import timedelta

# 1. PRE-CALCULATE TOTAL TRANSACTIONS
# Estimate total rows: (Days * Stores * Avg Daily Transactions)
total_days = (END_DATE - START_DATE).days + 1
date_range = pd.date_range(START_DATE, END_DATE)
# Total rows roughly = days * num_stores * 125 (midpoint of 50-200)
approx_rows = total_days * len(df_stores) * 125 

# 2. VECTORIZED TRANSACTION GENERATION
# Generate base transaction table using numpy for speed
transaction_data = {
    "transaction_id": np.arange(1, approx_rows + 1),
    "date": np.random.choice(date_range, approx_rows),
    "store_id": np.random.choice(df_stores["store_id"], approx_rows),
    "customer_id": np.random.choice(df_customers["customer_id"], approx_rows),
    "sku": np.random.choice(df_products["sku"], approx_rows),
    "quantity": np.random.randint(1, 5, approx_rows)
}
df_transactions = pd.DataFrame(transaction_data)

# 3. VECTORIZED PRODUCT JOIN
# Join with product data to get selling_price in one go
df_transactions = df_transactions.merge(
    df_products[["sku", "selling_price"]], on="sku", how="left"
).rename(columns={"selling_price": "unit_price"})

# 4. VECTORIZED PROMOTION MATCHING
# Merge with promotions where sku matches
df_transactions = df_transactions.merge(df_promotions, on="sku", how="left")

# Apply promotion logic: only keep discounts where transaction date is within promo range
# Use np.where for fast conditional logic
mask = (df_transactions["date"] >= df_transactions["start_date"]) & \
       (df_transactions["date"] <= df_transactions["end_date"])

df_transactions["discount_pct"] = np.where(mask, df_transactions["discount_pct"], 0)

# 5. FINAL CALCULATIONS (Vectorized)
df_transactions["total_sales"] = (
    df_transactions["unit_price"] * 
    df_transactions["quantity"] * 
    (1 - df_transactions["discount_pct"] / 100)
).round(2)

# Clean up temporary columns from merge
df_transactions = df_transactions.drop(columns=["start_date", "end_date", "promo_id"])


In [5]:
# SAVE ALL FILES
# ----------------------------
df_stores.to_csv("../Data/stores.csv", index=False)
df_products.to_csv("../Data/products.csv", index=False)
df_customers.to_csv("../Data/customers.csv", index=False)
df_employees.to_csv("../Data/employees.csv", index=False)
df_inventory.to_csv("../Data/inventory.csv", index=False)
df_promotions.to_csv("../Data/promotions.csv", index=False)
df_transactions.to_csv("../Data/transactions.csv", index=False)

print("Synthetic retail dataset generated successfully.")

Synthetic retail dataset generated successfully.
